In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:

import os
import torch

DATA_DIR = (
    "/kaggle/input/datasets/uraninjo/"
    "augmented-alzheimer-mri-dataset"
)

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

if not os.path.exists(DATA_DIR):
    print("Dataset path not found. Available inputs:")
    print(os.listdir("/kaggle/input"))
else:
    for folder in sorted(os.listdir(DATA_DIR)):
        path = os.path.join(DATA_DIR, folder)
        if os.path.isdir(path):
            counts = {}
            for cls in sorted(os.listdir(path)):
                class_path = os.path.join(path, cls)
                if os.path.isdir(class_path):
                    counts[cls] = sum(
                        f.lower().endswith((".jpg", ".jpeg", ".png"))
                        for f in os.listdir(class_path)
                    )
            print(f"\n{folder}:")
            print(counts)


PyTorch: 2.11.0+cu128
GPU available: True
GPU: Tesla T4

AugmentedAlzheimerDataset:
{'MildDemented': 8960, 'ModerateDemented': 6464, 'NonDemented': 9600, 'VeryMildDemented': 8960}

OriginalDataset:
{'MildDemented': 896, 'ModerateDemented': 64, 'NonDemented': 3200, 'VeryMildDemented': 2240}


In [2]:

import os
import random
from collections import defaultdict
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)

DATA_DIR = (
    "/kaggle/input/datasets/uraninjo/"
    "augmented-alzheimer-mri-dataset/AugmentedAlzheimerDataset"
)

CLASSES = [
    "MildDemented",
    "ModerateDemented",
    "NonDemented",
    "VeryMildDemented",
]

MAX_PER_CLASS = 1000

paths, labels = [], []

for label, cls in enumerate(CLASSES):
    folder = os.path.join(DATA_DIR, cls)
    files = [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]
    random.shuffle(files)
    chosen = files[:MAX_PER_CLASS]

    paths.extend(chosen)
    labels.extend([label] * len(chosen))

print("Selected images:", len(paths))
print("Class counts:", {
    cls: sum(y == i for y in labels)
    for i, cls in enumerate(CLASSES)
})

# 80% train, 20% temporary
train_paths, temp_paths, train_y, temp_y = train_test_split(
    paths, labels,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

# Split remaining 20% equally into validation and test
val_paths, test_paths, val_y, test_y = train_test_split(
    temp_paths, temp_y,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_y
)

print(f"Train: {len(train_paths)}")
print(f"Validation: {len(val_paths)}")
print(f"Test: {len(test_paths)}")
print("Classes:", CLASSES)


Selected images: 4000
Class counts: {'MildDemented': 1000, 'ModerateDemented': 1000, 'NonDemented': 1000, 'VeryMildDemented': 1000}
Train: 3200
Validation: 400
Test: 400
Classes: ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']


In [3]:

import time
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class MRIDataset(Dataset):
    def __init__(self, paths, labels, train=False):
        self.paths = paths
        self.labels = labels
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5) if train
                else transforms.Lambda(lambda x: x),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as img:
            img = img.convert("RGB")
            img = self.transform(img)
        return img, self.labels[idx]

BATCH_SIZE = 32
NUM_WORKERS = 2

train_ds = MRIDataset(train_paths, train_y, train=True)
val_ds = MRIDataset(val_paths, val_y)
test_ds = MRIDataset(test_paths, test_y)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

start = time.time()
images, labels = next(iter(train_loader))

print("Batch shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image dtype:", images.dtype)
print("GPU:", torch.cuda.get_device_name(0))
print(f"First batch loading: {time.time()-start:.2f} seconds")
print("Train / validation / test:",
      len(train_ds), len(val_ds), len(test_ds))


Batch shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Image dtype: torch.float32
GPU: Tesla T4
First batch loading: 1.32 seconds
Train / validation / test: 3200 400 400


In [4]:

import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained ResNet18
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

# Freeze early layers; fine-tune layer4 and the classifier
for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

model.fc = nn.Sequential(
    nn.Dropout(0.25),
    nn.Linear(model.fc.in_features, 4)
)

model = model.to(DEVICE)

# AdamW with separate learning rates
optimizer = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 5e-4}
], weight_decay=1e-4)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=1
)

scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

print("Model: ResNet18")
print("Device:", DEVICE)
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters:",
      sum(p.numel() for p in model.parameters()))
print("Mixed precision enabled:", DEVICE.type == "cuda")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 189MB/s]


Model: ResNet18
Device: cuda
Trainable parameters: 8395780
Total parameters: 11178564
Mixed precision enabled: True


In [6]:

import time
import torch
from sklearn.metrics import accuracy_score

def run_train_epoch(model, loader, optimizer, criterion, scaler, max_seconds=170):
    model.train()
    start = time.time()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        if time.time() - start > max_seconds:
            print("Time guard reached; stopping this epoch.")
            break

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(labels)
        all_preds.extend(logits.argmax(1).detach().cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    accuracy = accuracy_score(all_labels, all_preds) if all_labels else 0
    print(f"Train accuracy on processed batches: {accuracy*100:.2f}%")
    print(f"Processed: {len(all_labels)}/{len(loader.dataset)} images")
    print(f"Elapsed: {time.time()-start:.1f}s")
    return len(all_labels) == len(loader.dataset)

completed = run_train_epoch(
    model, train_loader, optimizer, criterion, scaler
)


Train accuracy on processed batches: 76.84%
Processed: 3200/3200 images
Elapsed: 7.8s


In [8]:
from sklearn.metrics import classification_report
import torch

classes = [
    "MildDemented",
    "ModerateDemented",
    "NonDemented",
    "VeryMildDemented"
]

@torch.no_grad()
def evaluate_report(model, loader):
    model.eval()
    y_true, y_pred = [], []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(images)

        y_pred.extend(logits.argmax(1).cpu().tolist())
        y_true.extend(labels.tolist())

    print(classification_report(
        y_true, y_pred,
        labels=[0, 1, 2, 3],
        target_names=classes,
        digits=3,
        zero_division=0
    ))

evaluate_report(model, val_loader)

                  precision    recall  f1-score   support

    MildDemented      0.700     0.630     0.663       100
ModerateDemented      0.980     1.000     0.990       100
     NonDemented      0.643     0.720     0.679       100
VeryMildDemented      0.458     0.440     0.449       100

        accuracy                          0.698       400
       macro avg      0.695     0.697     0.695       400
    weighted avg      0.695     0.698     0.695       400



In [9]:
# Train one additional epoch
completed = run_train_epoch(
    model, train_loader, optimizer, criterion, scaler,
    max_seconds=170
)

# Evaluate validation performance
val_acc = evaluate(model, val_loader)

# Update learning rate using validation accuracy
scheduler.step(val_acc)

print(f"Updated validation accuracy: {val_acc * 100:.2f}%")

Train accuracy on processed batches: 85.34%
Processed: 3200/3200 images
Elapsed: 7.6s
Validation accuracy: 71.75%
                  precision    recall  f1-score   support

    MildDemented      0.656     0.800     0.721       100
ModerateDemented      0.990     1.000     0.995       100
     NonDemented      0.654     0.680     0.667       100
VeryMildDemented      0.534     0.390     0.451       100

        accuracy                          0.718       400
       macro avg      0.708     0.718     0.708       400
    weighted avg      0.708     0.718     0.708       400

Updated validation accuracy: 71.75%


In [10]:
import copy

best_acc = val_acc
best_state = copy.deepcopy(model.state_dict())
patience = 2
wait = 0

for epoch in range(3, 8):
    print(f"\nEpoch {epoch}")

    completed = run_train_epoch(
        model, train_loader, optimizer, criterion, scaler,
        max_seconds=170
    )

    if not completed:
        print("Epoch incomplete. Stopping.")
        break

    current_acc = evaluate(model, val_loader)
    scheduler.step(current_acc)

    if current_acc > best_acc:
        best_acc = current_acc
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
        print("Best model updated!")
    else:
        wait += 1

    if wait >= patience:
        print("Early stopping triggered.")
        break

model.load_state_dict(best_state)

print(f"\nBest validation accuracy: {best_acc * 100:.2f}%")
torch.save(model.state_dict(), "/kaggle/working/best_resnet18.pth")


Epoch 3
Train accuracy on processed batches: 92.09%
Processed: 3200/3200 images
Elapsed: 7.1s
Validation accuracy: 76.25%
                  precision    recall  f1-score   support

    MildDemented      0.725     0.790     0.756       100
ModerateDemented      0.990     1.000     0.995       100
     NonDemented      0.780     0.640     0.703       100
VeryMildDemented      0.574     0.620     0.596       100

        accuracy                          0.762       400
       macro avg      0.767     0.763     0.763       400
    weighted avg      0.767     0.762     0.763       400

Best model updated!

Epoch 4
Train accuracy on processed batches: 95.06%
Processed: 3200/3200 images
Elapsed: 7.0s
Validation accuracy: 74.25%
                  precision    recall  f1-score   support

    MildDemented      0.763     0.710     0.736       100
ModerateDemented      0.990     1.000     0.995       100
     NonDemented      0.687     0.680     0.683       100
VeryMildDemented      0.542     0.

In [11]:
import torch
import numpy as np
from torchvision import models
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import time

# Rebuild the same ResNet18 architecture
feature_model = models.resnet18(weights=None)
feature_model.fc = nn.Sequential(
    nn.Dropout(0.25),
    nn.Linear(feature_model.fc.in_features, 4)
)
feature_model.load_state_dict(
    torch.load(
        "/kaggle/working/best_resnet18.pth",
        map_location=DEVICE,
        weights_only=True
    )
)

# Remove classifier to output 512-D features
feature_model.fc = nn.Identity()
feature_model = feature_model.to(DEVICE).eval()

@torch.no_grad()
def extract_features(loader):
    features, labels = [], []
    start = time.time()

    for images, y in loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            z = feature_model(images)

        features.append(z.float().cpu())
        labels.append(y.cpu())

    X = torch.cat(features)
    y = torch.cat(labels)
    print(f"Extracted {len(y)} samples in {time.time()-start:.1f}s")
    return X, y

train_features, train_labels = extract_features(train_loader)
val_features, val_labels = extract_features(val_loader)
test_features, test_labels = extract_features(test_loader)

torch.save({
    "X_train": train_features,
    "y_train": train_labels,
    "X_val": val_features,
    "y_val": val_labels,
    "X_test": test_features,
    "y_test": test_labels,
}, "/kaggle/working/cached_features.pt")

print("Feature shapes:", train_features.shape,
      val_features.shape, test_features.shape)
print("Saved cached features.")

Extracted 3200 samples in 7.2s
Extracted 400 samples in 1.0s
Extracted 400 samples in 3.3s
Feature shapes: torch.Size([3200, 512]) torch.Size([400, 512]) torch.Size([400, 512])
Saved cached features.


In [12]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# Load cached features
data = torch.load(
    "/kaggle/working/cached_features.pt",
    map_location="cpu",
    weights_only=True
)

X_train = data["X_train"].float()
y_train = data["y_train"].long()
X_val = data["X_val"].float()
y_val = data["y_val"].long()
X_test = data["X_test"].float()
y_test = data["y_test"].long()

# Fit standardization using training features only
feature_mean = X_train.mean(dim=0, keepdim=True)
feature_std = X_train.std(dim=0, keepdim=True).clamp_min(1e-6)

X_train_scaled = (X_train - feature_mean) / feature_std
X_val_scaled = (X_val - feature_mean) / feature_std
X_test_scaled = (X_test - feature_mean) / feature_std

# Small learned projection: 512 -> 64 -> 8
projection = nn.Sequential(
    nn.Linear(512, 64),
    nn.LayerNorm(64),
    nn.GELU(),
    nn.Dropout(0.15),
    nn.Linear(64, 8)
)

print("Scaled feature shapes:")
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)
print("Projection:", "512 -> 64 -> 8")
print("Feature means/stds fitted on training data only.")

Scaled feature shapes:
Train: torch.Size([3200, 512])
Validation: torch.Size([400, 512])
Test: torch.Size([400, 512])
Projection: 512 -> 64 -> 8
Feature means/stds fitted on training data only.


In [14]:
%pip install -q pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 55.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 22.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 75.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 110.8 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [15]:
import pennylane as qml
import torch

print("PennyLane:", qml.__version__)
print("Torch:", torch.__version__)

dev = qml.device("default.qubit", wires=8)
print("8-qubit simulator ready!")

PennyLane: 0.45.1
Torch: 2.11.0+cu128
8-qubit simulator ready!


In [16]:
import pennylane as qml
import torch

print("PennyLane:", qml.__version__)
print("Torch:", torch.__version__)

n_qubits = 8
n_layers = 2

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(
        inputs, wires=range(n_qubits), rotation="Y"
    )
    qml.StronglyEntanglingLayers(
        weights, wires=range(n_qubits)
    )
    return [
        qml.expval(qml.PauliZ(i))
        for i in range(n_qubits)
    ]

weight_shapes = {
    "weights": (n_layers, n_qubits, 3)
}

print("Qubits:", n_qubits)
print("Entangling layers:", n_layers)
print("Trainable circuit parameters:", n_layers * n_qubits * 3)
print("Circuit initialized successfully.")

PennyLane: 0.45.1
Torch: 2.11.0+cu128
Qubits: 8
Entangling layers: 2
Trainable circuit parameters: 48
Circuit initialized successfully.


In [17]:
import torch
from torch import nn
import pennylane as qml

N_QUBITS = 8
N_LAYERS = 2
N_CLASSES = 4

dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(
        inputs, wires=range(N_QUBITS), rotation="Y"
    )
    qml.StronglyEntanglingLayers(
        weights, wires=range(N_QUBITS)
    )
    return [
        qml.expval(qml.PauliZ(i))
        for i in range(N_QUBITS)
    ]

class HybridQuantumClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.projection = nn.Sequential(
            nn.Linear(512, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, 8)
        )

        self.quantum_weights = nn.Parameter(
            0.01 * torch.randn(N_LAYERS, N_QUBITS, 3)
        )

        self.output_head = nn.Linear(8, N_CLASSES)

    def forward(self, x):
        angles = self.projection(x)
        # Bound RY angles to [-pi, pi]
        angles = torch.pi * torch.tanh(angles)

        q_out = torch.stack([
            torch.stack(quantum_circuit(
                sample, self.quantum_weights
            ))
            for sample in angles
        ])

        return self.output_head(q_out)

quantum_model = HybridQuantumClassifier()

print("Hybrid model initialized.")
print("Projection: 512 -> 64 -> 8")
print("Quantum circuit parameters:", quantum_model.quantum_weights.numel())
print("Output classes:", N_CLASSES)
print("Quantum device: default.qubit")

Hybrid model initialized.
Projection: 512 -> 64 -> 8
Quantum circuit parameters: 48
Output classes: 4
Quantum device: default.qubit


In [18]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn

BATCH_SIZE_Q = 8

train_q_loader = DataLoader(
    TensorDataset(X_train_scaled, y_train),
    batch_size=BATCH_SIZE_Q,
    shuffle=True
)

val_q_loader = DataLoader(
    TensorDataset(X_val_scaled, y_val),
    batch_size=BATCH_SIZE_Q,
    shuffle=False
)

optimizer_q = torch.optim.AdamW(
    quantum_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

criterion_q = nn.CrossEntropyLoss()

print("Quantum train batches:", len(train_q_loader))
print("Quantum validation batches:", len(val_q_loader))
print("Optimizer: AdamW, learning rate 0.001")
print("Batch size:", BATCH_SIZE_Q)

Quantum train batches: 400
Quantum validation batches: 50
Optimizer: AdamW, learning rate 0.001
Batch size: 8


In [21]:
def forward(self, x):
    angles = self.projection(x)
    angles = torch.pi * torch.tanh(angles)

    q_out = torch.stack([
        torch.stack(
            quantum_circuit(sample, self.quantum_weights)
        )
        for sample in angles
    ])

    # PennyLane may return float64; match PyTorch layer dtype
    q_out = q_out.to(dtype=self.output_head.weight.dtype)

    return self.output_head(q_out)

In [22]:
quantum_model = HybridQuantumClassifier()

optimizer_q = torch.optim.AdamW(
    quantum_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

criterion_q = nn.CrossEntropyLoss()

print("Model and optimizer recreated.")

Model and optimizer recreated.


In [24]:
import torch
from torch import nn
import pennylane as qml

N_QUBITS = 8
N_LAYERS = 2
N_CLASSES = 4

dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(
        inputs, wires=range(N_QUBITS), rotation="Y"
    )
    qml.StronglyEntanglingLayers(
        weights, wires=range(N_QUBITS)
    )
    return [
        qml.expval(qml.PauliZ(i))
        for i in range(N_QUBITS)
    ]


class HybridQuantumClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.projection = nn.Sequential(
            nn.Linear(512, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, 8)
        )

        self.quantum_weights = nn.Parameter(
            0.01 * torch.randn(2, 8, 3)
        )

        self.output_head = nn.Linear(8, 4)

    def forward(self, x):
        angles = self.projection(x)
        angles = torch.pi * torch.tanh(angles)

        outputs = []
        for sample in angles:
            result = quantum_circuit(
                sample, self.quantum_weights
            )
            outputs.append(torch.stack(list(result)))

        q_out = torch.stack(outputs)

        # Explicitly match the output layer's dtype
        q_out = q_out.to(
            device=self.output_head.weight.device,
            dtype=self.output_head.weight.dtype
        )

        return self.output_head(q_out)


# Recreate model and optimizer
quantum_model = HybridQuantumClassifier()

optimizer_q = torch.optim.AdamW(
    quantum_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

criterion_q = nn.CrossEntropyLoss()

print("Model redefined and recreated.")
print("Output layer dtype:", quantum_model.output_head.weight.dtype)

Model redefined and recreated.
Output layer dtype: torch.float32


In [25]:
import time

quantum_model.train()
xb, yb = next(iter(train_q_loader))

start = time.time()
optimizer_q.zero_grad(set_to_none=True)

logits = quantum_model(xb)
loss = criterion_q(logits, yb)
loss.backward()
optimizer_q.step()

print("Logits shape:", logits.shape)
print(f"Batch loss: {loss.item():.4f}")
print(f"Batch accuracy: {(logits.argmax(1) == yb).float().mean().item()*100:.1f}%")
print(f"Forward + backward time: {time.time()-start:.2f}s")

Logits shape: torch.Size([8, 4])
Batch loss: 1.4569
Batch accuracy: 25.0%
Forward + backward time: 0.40s


In [26]:
import time
import torch

MAX_SECONDS = 60
start = time.time()
processed = 0
loss_sum = 0.0
correct = 0

quantum_model.train()

for xb, yb in train_q_loader:
    if time.time() - start >= MAX_SECONDS:
        print("Time limit reached.")
        break

    optimizer_q.zero_grad(set_to_none=True)

    logits = quantum_model(xb)
    loss = criterion_q(logits, yb)
    loss.backward()
    optimizer_q.step()

    processed += len(yb)
    loss_sum += loss.item() * len(yb)
    correct += (logits.argmax(1) == yb).sum().item()

print(f"Processed: {processed} training samples")
print(f"Subset training accuracy: {100*correct/max(processed,1):.2f}%")
print(f"Mean loss: {loss_sum/max(processed,1):.4f}")
print(f"Elapsed: {time.time()-start:.1f}s")

torch.save(
    quantum_model.state_dict(),
    "/kaggle/working/quantum_pilot.pth"
)
print("Pilot checkpoint saved.")

Time limit reached.
Processed: 1552 training samples
Subset training accuracy: 67.85%
Mean loss: 0.9830
Elapsed: 60.2s
Pilot checkpoint saved.


In [27]:
import torch
from sklearn.metrics import accuracy_score, classification_report

quantum_model.eval()
val_preds, val_true = [], []

with torch.no_grad():
    for xb, yb in val_q_loader:
        logits = quantum_model(xb)
        val_preds.extend(logits.argmax(1).cpu().tolist())
        val_true.extend(yb.tolist())

val_q_acc = accuracy_score(val_true, val_preds)

print(f"Quantum validation accuracy: {val_q_acc*100:.2f}%")
print(classification_report(
    val_true,
    val_preds,
    labels=[0, 1, 2, 3],
    target_names=classes,
    digits=3,
    zero_division=0
))

Quantum validation accuracy: 65.00%
                  precision    recall  f1-score   support

    MildDemented      0.740     0.770     0.755       100
ModerateDemented      0.952     1.000     0.976       100
     NonDemented      0.000     0.000     0.000       100
VeryMildDemented      0.435     0.830     0.570       100

        accuracy                          0.650       400
       macro avg      0.532     0.650     0.575       400
    weighted avg      0.532     0.650     0.575       400



In [28]:
import torch
from sklearn.metrics import accuracy_score, classification_report

quantum_model.eval()
val_preds, val_true = [], []

with torch.no_grad():
    for xb, yb in val_q_loader:
        logits = quantum_model(xb)
        val_preds.extend(logits.argmax(1).cpu().tolist())
        val_true.extend(yb.tolist())

val_q_acc = accuracy_score(val_true, val_preds)

print(f"Quantum validation accuracy: {val_q_acc*100:.2f}%")
print(classification_report(
    val_true,
    val_preds,
    labels=[0, 1, 2, 3],
    target_names=classes,
    digits=3,
    zero_division=0
))

Quantum validation accuracy: 65.00%
                  precision    recall  f1-score   support

    MildDemented      0.740     0.770     0.755       100
ModerateDemented      0.952     1.000     0.976       100
     NonDemented      0.000     0.000     0.000       100
VeryMildDemented      0.435     0.830     0.570       100

        accuracy                          0.650       400
       macro avg      0.532     0.650     0.575       400
    weighted avg      0.532     0.650     0.575       400



In [29]:
import numpy as np
from collections import Counter

print("True class counts:")
print(Counter(val_true))

print("\nPredicted class counts:")
print(Counter(val_preds))

print("\nClass order:")
for i, name in enumerate(classes):
    print(i, name)

True class counts:
Counter({2: 100, 1: 100, 0: 100, 3: 100})

Predicted class counts:
Counter({3: 191, 1: 105, 0: 104})

Class order:
0 MildDemented
1 ModerateDemented
2 NonDemented
3 VeryMildDemented


In [30]:
import torch
from torch import nn
import time

# Fresh model to avoid carrying forward the collapsed pilot
quantum_model = HybridQuantumClassifier()

optimizer_q = torch.optim.AdamW(
    quantum_model.parameters(),
    lr=0.0002,
    weight_decay=1e-4
)

# Calculate weights from training labels
counts = torch.bincount(y_train, minlength=4).float()
class_weights = (counts.sum() / (4 * counts)).float()

criterion_q = nn.CrossEntropyLoss(weight=class_weights)

print("Training class counts:", counts.tolist())
print("Loss weights:", class_weights.tolist())
print("Learning rate:", 0.0002)

Training class counts: [800.0, 800.0, 800.0, 800.0]
Loss weights: [1.0, 1.0, 1.0, 1.0]
Learning rate: 0.0002


In [31]:
start = time.time()
processed, correct, loss_sum = 0, 0, 0.0
quantum_model.train()

for xb, yb in train_q_loader:
    if time.time() - start >= 60:
        print("Time limit reached.")
        break

    optimizer_q.zero_grad(set_to_none=True)
    logits = quantum_model(xb)
    loss = criterion_q(logits, yb)
    loss.backward()
    optimizer_q.step()

    processed += len(yb)
    correct += (logits.argmax(1) == yb).sum().item()
    loss_sum += loss.item() * len(yb)

print(f"Processed: {processed}")
print(f"Training subset accuracy: {100*correct/max(processed,1):.2f}%")
print(f"Mean weighted loss: {loss_sum/max(processed,1):.4f}")
print(f"Elapsed: {time.time()-start:.1f}s")

Time limit reached.
Processed: 1560
Training subset accuracy: 56.15%
Mean weighted loss: 1.2398
Elapsed: 60.3s


In [32]:
import torch
from sklearn.metrics import accuracy_score, classification_report

# Rebuild the same ResNet18 architecture
cnn_test_model = models.resnet18(weights=None)
cnn_test_model.fc = nn.Sequential(
    nn.Dropout(0.25),
    nn.Linear(cnn_test_model.fc.in_features, 4)
)

cnn_test_model.load_state_dict(
    torch.load(
        "/kaggle/working/best_resnet18.pth",
        map_location=DEVICE,
        weights_only=True
    )
)
cnn_test_model = cnn_test_model.to(DEVICE).eval()

y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = cnn_test_model(images)

        y_pred.extend(logits.argmax(1).cpu().tolist())
        y_true.extend(labels.tolist())

cnn_test_acc = accuracy_score(y_true, y_pred)

print(f"CNN TEST ACCURACY: {cnn_test_acc * 100:.2f}%")
print(classification_report(
    y_true, y_pred,
    labels=[0, 1, 2, 3],
    target_names=classes,
    digits=3,
    zero_division=0
))

CNN TEST ACCURACY: 79.50%
                  precision    recall  f1-score   support

    MildDemented      0.776     0.830     0.802       100
ModerateDemented      0.990     0.990     0.990       100
     NonDemented      0.821     0.640     0.719       100
VeryMildDemented      0.626     0.720     0.670       100

        accuracy                          0.795       400
       macro avg      0.803     0.795     0.795       400
    weighted avg      0.803     0.795     0.795       400



In [33]:

import time, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tf_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
tf_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class MRIDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.transform = paths, labels, transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), int(self.labels[i])

tr_ds = MRIDataset(train_paths, train_y, tf_train)
va_ds = MRIDataset(val_paths, val_y, tf_eval)

tr_dl = DataLoader(tr_ds, batch_size=32, shuffle=True,
                   num_workers=2, pin_memory=True)
va_dl = DataLoader(va_ds, batch_size=32, shuffle=False,
                   num_workers=2, pin_memory=True)

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
for p in model.parameters():
    p.requires_grad = False

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 4)
model = model.to(device)

optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

start = time.time()
model.train()
for x, y in tr_dl:
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()

model.eval()
preds, targets = [], []
with torch.no_grad():
    for x, y in va_dl:
        out = model(x.to(device))
        preds.extend(out.argmax(1).cpu().tolist())
        targets.extend(y.tolist())

print(f"EfficientNet-B0 | 1 epoch")
print(f"Validation accuracy: {accuracy_score(targets, preds)*100:.2f}%")
print(f"Time: {(time.time()-start)/60:.2f} minutes")


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 160MB/s]


EfficientNet-B0 | 1 epoch
Validation accuracy: 50.25%
Time: 0.19 minutes


In [34]:

from sklearn.metrics import accuracy_score
import time

optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=5e-4)

for epoch in range(2, 6):
    start = time.time()
    model.train()
    total_loss = 0

    for x, y in tr_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in va_dl:
            out = model(x.to(device))
            preds.extend(out.argmax(1).cpu().tolist())
            targets.extend(y.tolist())

    acc = accuracy_score(targets, preds) * 100
    print(f"Epoch {epoch}/5 | Loss: {total_loss/len(tr_dl):.4f} "
          f"| Val Accuracy: {acc:.2f}% "
          f"| Time: {(time.time()-start):.1f}s")


Epoch 2/5 | Loss: 1.0988 | Val Accuracy: 52.50% | Time: 8.7s
Epoch 3/5 | Loss: 1.0580 | Val Accuracy: 52.25% | Time: 8.8s
Epoch 4/5 | Loss: 1.0286 | Val Accuracy: 54.50% | Time: 8.8s
Epoch 5/5 | Loss: 1.0085 | Val Accuracy: 57.00% | Time: 8.8s


In [35]:

# Unfreeze the final EfficientNet feature block
for p in model.features[-1].parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": model.features[-1].parameters(), "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 2e-4}
], weight_decay=1e-4)

for epoch in range(6, 11):
    start = time.time()
    model.train()
    total_loss = 0

    for x, y in tr_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in va_dl:
            out = model(x.to(device))
            preds.extend(out.argmax(1).cpu().tolist())
            targets.extend(y.tolist())

    acc = accuracy_score(targets, preds) * 100
    print(f"Epoch {epoch}/10 | Loss: {total_loss/len(tr_dl):.4f} "
          f"| Val Accuracy: {acc:.2f}% "
          f"| Time: {time.time()-start:.1f}s")


Epoch 6/10 | Loss: 0.9823 | Val Accuracy: 56.00% | Time: 8.9s
Epoch 7/10 | Loss: 0.9486 | Val Accuracy: 58.50% | Time: 8.8s
Epoch 8/10 | Loss: 0.9183 | Val Accuracy: 58.50% | Time: 8.7s
Epoch 9/10 | Loss: 0.8980 | Val Accuracy: 60.50% | Time: 8.9s
Epoch 10/10 | Loss: 0.8757 | Val Accuracy: 61.00% | Time: 8.7s


In [36]:

# Unfreeze the last two feature blocks
for block in model.features[-2:]:
    for p in block.parameters():
        p.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": model.features[-2:].parameters(), "lr": 3e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4}
], weight_decay=1e-4)

best_effnet_val = 0.0

for epoch in range(11, 16):
    start = time.time()
    model.train()
    total_loss = 0

    for x, y in tr_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in va_dl:
            out = model(x.to(device))
            preds.extend(out.argmax(1).cpu().tolist())
            targets.extend(y.tolist())

    acc = accuracy_score(targets, preds) * 100
    if acc > best_effnet_val:
        best_effnet_val = acc
        torch.save(model.state_dict(), "/kaggle/working/best_effnet.pth")

    print(f"Epoch {epoch}/15 | Loss: {total_loss/len(tr_dl):.4f} "
          f"| Val Accuracy: {acc:.2f}% "
          f"| Time: {time.time()-start:.1f}s")


Epoch 11/15 | Loss: 0.8546 | Val Accuracy: 61.25% | Time: 8.9s
Epoch 12/15 | Loss: 0.8176 | Val Accuracy: 60.50% | Time: 8.9s
Epoch 13/15 | Loss: 0.8201 | Val Accuracy: 62.50% | Time: 9.0s
Epoch 14/15 | Loss: 0.7941 | Val Accuracy: 62.25% | Time: 8.9s
Epoch 15/15 | Loss: 0.7691 | Val Accuracy: 62.50% | Time: 8.8s


In [37]:

import torch
from sklearn.metrics import accuracy_score, classification_report

model.load_state_dict(torch.load(
    "/kaggle/working/best_effnet.pth",
    map_location=device,
    weights_only=True
))
model.eval()

te_ds = MRIDataset(test_paths, test_y, tf_eval)
te_dl = DataLoader(te_ds, batch_size=32, shuffle=False, num_workers=2)

preds, targets = [], []
with torch.no_grad():
    for x, y in te_dl:
        out = model(x.to(device))
        preds.extend(out.argmax(1).cpu().tolist())
        targets.extend(y.tolist())

print(f"EfficientNet test accuracy: {accuracy_score(targets, preds)*100:.2f}%")
print(classification_report(
    targets, preds,
    labels=[0, 1, 2, 3],
    target_names=["MildDemented", "ModerateDemented",
                  "NonDemented", "VeryMildDemented"],
    zero_division=0
))


EfficientNet test accuracy: 62.75%
                  precision    recall  f1-score   support

    MildDemented       0.61      0.64      0.62       100
ModerateDemented       0.79      0.92      0.85       100
     NonDemented       0.63      0.55      0.59       100
VeryMildDemented       0.43      0.40      0.42       100

        accuracy                           0.63       400
       macro avg       0.62      0.63      0.62       400
    weighted avg       0.62      0.63      0.62       400



In [39]:

import torch
import torch.nn as nn
import copy, time
from sklearn.metrics import accuracy_score

# Unfreeze the final two feature blocks
for p in model.parameters():
    p.requires_grad = False

for p in model.features[-2:].parameters():
    p.requires_grad = True

for p in model.classifier.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": model.features[-2:].parameters(), "lr": 2e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4}
], weight_decay=1e-4)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
best_acc = 0.0
best_state = copy.deepcopy(model.state_dict())

for epoch in range(1, 6):
    start = time.time()
    model.train()
    total_loss = 0.0

    for x, y in tr_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in va_dl:
            preds.extend(model(x.to(device)).argmax(1).cpu().tolist())
            targets.extend(y.tolist())

    acc = accuracy_score(targets, preds)
    if acc > best_acc:
        best_acc = acc
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, "/kaggle/working/best_effnet_tuned.pth")

    print(f"Epoch {epoch}/5 | Loss {total_loss/len(tr_dl):.4f} "
          f"| Val accuracy {acc*100:.2f}% "
          f"| Best {best_acc*100:.2f}% "
          f"| {time.time()-start:.1f}s")

model.load_state_dict(best_state)


Epoch 1/5 | Loss 0.8649 | Val accuracy 62.50% | Best 62.50% | 9.5s
Epoch 2/5 | Loss 0.8439 | Val accuracy 62.50% | Best 62.50% | 9.8s
Epoch 3/5 | Loss 0.8399 | Val accuracy 62.25% | Best 62.50% | 9.6s
Epoch 4/5 | Loss 0.8229 | Val accuracy 62.00% | Best 62.50% | 9.2s
Epoch 5/5 | Loss 0.8206 | Val accuracy 62.50% | Best 62.50% | 9.2s


<All keys matched successfully>

In [40]:

import os

for f in [
    "/kaggle/working/best_resnet18.pth",
    "/kaggle/working/cached_features.pt",
    "/kaggle/working/best_resnet18_tuned.pth"
]:
    print(f, "EXISTS" if os.path.exists(f) else "NOT FOUND")


/kaggle/working/best_resnet18.pth EXISTS
/kaggle/working/cached_features.pt EXISTS
/kaggle/working/best_resnet18_tuned.pth NOT FOUND


In [42]:

import torch

print("Available variables:")
for name in ["train_features", "val_features", "test_features",
             "X_train_scaled", "X_val_scaled", "X_test_scaled",
             "y_train", "y_val", "y_test", "va_dl", "val_dl"]:
    print(f"{name}: {'YES' if name in globals() else 'NO'}")

cached = torch.load(
    "/kaggle/working/cached_features.pt",
    map_location="cpu",
    weights_only=False
)
print("\nCached file type:", type(cached))
if isinstance(cached, dict):
    for k, v in cached.items():
        print(k, tuple(v.shape) if torch.is_tensor(v) else type(v))


Available variables:
train_features: YES
val_features: YES
test_features: YES
X_train_scaled: YES
X_val_scaled: YES
X_test_scaled: YES
y_train: YES
y_val: YES
y_test: YES
va_dl: YES
val_dl: NO

Cached file type: <class 'dict'>
X_train (3200, 512)
y_train (3200,)
X_val (400, 512)
y_val (400,)
X_test (400, 512)
y_test (400,)


In [43]:

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np
import joblib
import time

start = time.time()
best_svm, best_acc, best_params = None, 0, None

for c in [1, 10, 50]:
    for gamma in ["scale", 0.001, 0.01]:
        clf = SVC(C=c, gamma=gamma, kernel="rbf",
                  class_weight="balanced")
        clf.fit(X_train_scaled, y_train)
        val_pred = clf.predict(X_val_scaled)
        acc = accuracy_score(y_val, val_pred)

        print(f"C={c}, gamma={gamma}, val={acc*100:.2f}%")

        if acc > best_acc:
            best_svm, best_acc = clf, acc
            best_params = (c, gamma)

joblib.dump(best_svm, "/kaggle/working/best_resnet_feature_svm.pkl")

print(f"\nBest validation accuracy: {best_acc*100:.2f}%")
print("Best parameters:", best_params)
print(f"Time: {(time.time()-start):.1f}s")


C=1, gamma=scale, val=77.00%
C=1, gamma=0.001, val=76.50%
C=1, gamma=0.01, val=74.25%
C=10, gamma=scale, val=77.00%
C=10, gamma=0.001, val=76.75%
C=10, gamma=0.01, val=74.75%
C=50, gamma=scale, val=77.00%
C=50, gamma=0.001, val=76.75%
C=50, gamma=0.01, val=74.75%

Best validation accuracy: 77.00%
Best parameters: (1, 'scale')
Time: 14.3s


In [44]:

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import ParameterGrid
import joblib, time

start = time.time()
best_svm, best_acc, best_params = None, 0.0, None

params = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.0001, 0.0005, 0.001, 0.005]
}

for p in ParameterGrid(params):
    clf = SVC(kernel="rbf", class_weight="balanced", **p)
    clf.fit(X_train_scaled, y_train)
    pred = clf.predict(X_val_scaled)
    acc = accuracy_score(y_val, pred)

    if acc > best_acc:
        best_svm, best_acc, best_params = clf, acc, p

print("Best validation accuracy:", round(best_acc*100, 2), "%")
print("Best parameters:", best_params)
print("Time:", round(time.time()-start, 1), "seconds")

joblib.dump(best_svm, "/kaggle/working/best_resnet_feature_svm.pkl")


Best validation accuracy: 78.25 %
Best parameters: {'C': 1, 'gamma': 0.005}
Time: 18.6 seconds


['/kaggle/working/best_resnet_feature_svm.pkl']

In [45]:

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import ParameterGrid
import joblib, time

start = time.time()
best_svm, best_acc, best_params = None, 0.0, None

params = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.0001, 0.0005, 0.001, 0.005]
}

for p in ParameterGrid(params):
    clf = SVC(kernel="rbf", class_weight="balanced", **p)
    clf.fit(X_train_scaled, y_train)
    pred = clf.predict(X_val_scaled)
    acc = accuracy_score(y_val, pred)

    if acc > best_acc:
        best_svm, best_acc, best_params = clf, acc, p

print("Best validation accuracy:", round(best_acc*100, 2), "%")
print("Best parameters:", best_params)
print("Time:", round(time.time()-start, 1), "seconds")

joblib.dump(best_svm, "/kaggle/working/best_resnet_feature_svm.pkl")


Best validation accuracy: 78.25 %
Best parameters: {'C': 1, 'gamma': 0.005}
Time: 20.1 seconds


['/kaggle/working/best_resnet_feature_svm.pkl']

In [46]:

import joblib
from sklearn.metrics import accuracy_score, classification_report

best_svm = joblib.load("/kaggle/working/best_resnet_feature_svm.pkl")
test_pred = best_svm.predict(X_test_scaled)

print(f"SVM Test Accuracy: {accuracy_score(y_test, test_pred)*100:.2f}%")
print(classification_report(
    y_test, test_pred,
    labels=[0, 1, 2, 3],
    target_names=[
        "MildDemented", "ModerateDemented",
        "NonDemented", "VeryMildDemented"
    ],
    zero_division=0
))


SVM Test Accuracy: 79.50%
                  precision    recall  f1-score   support

    MildDemented       0.80      0.78      0.79       100
ModerateDemented       1.00      0.99      0.99       100
     NonDemented       0.81      0.69      0.75       100
VeryMildDemented       0.61      0.72      0.66       100

        accuracy                           0.80       400
       macro avg       0.80      0.79      0.80       400
    weighted avg       0.80      0.80      0.80       400



In [48]:

for name in ["train_ds", "val_ds", "test_ds",
             "train_items", "val_items", "test_items",
             "train_paths", "val_paths", "test_paths",
             "train_dl", "va_dl", "tr_dl"]:
    print(f"{name}: {'EXISTS' if name in globals() else 'MISSING'}")


train_ds: EXISTS
val_ds: EXISTS
test_ds: EXISTS
train_items: MISSING
val_items: MISSING
test_items: MISSING
train_paths: EXISTS
val_paths: EXISTS
test_paths: EXISTS
train_dl: MISSING
va_dl: EXISTS
tr_dl: EXISTS


In [50]:

import torch
import torch.nn as nn
import copy, time
from sklearn.metrics import accuracy_score

# Restore original ResNet18 checkpoint
resnet = models.resnet18(weights=None)
resnet.fc = nn.Sequential(
    nn.Dropout(0.25),
    nn.Linear(resnet.fc.in_features, 4)
)
resnet.load_state_dict(torch.load(
    "/kaggle/working/best_resnet18.pth",
    map_location=device,
    weights_only=True
))
resnet = resnet.to(device)

for p in resnet.parameters():
    p.requires_grad = False
for p in resnet.layer3.parameters():
    p.requires_grad = True
for p in resnet.layer4.parameters():
    p.requires_grad = True
for p in resnet.fc.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": resnet.layer3.parameters(), "lr": 1e-5},
    {"params": resnet.layer4.parameters(), "lr": 3e-5},
    {"params": resnet.fc.parameters(), "lr": 1e-4}
], weight_decay=1e-4)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
best_acc = 0.0
best_state = copy.deepcopy(resnet.state_dict())

for epoch in range(1, 11):
    start = time.time()
    resnet.train()
    total_loss = 0.0

    for x, y in tr_dl:  # Correct loader
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(resnet(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    resnet.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in va_dl:
            preds.extend(resnet(x.to(device)).argmax(1).cpu().tolist())
            targets.extend(y.cpu().tolist())

    acc = accuracy_score(targets, preds)
    if acc > best_acc:
        best_acc = acc
        best_state = copy.deepcopy(resnet.state_dict())
        torch.save(best_state,
                   "/kaggle/working/best_resnet18_finetuned.pth")

    print(f"Epoch {epoch}/10 | Loss {total_loss/len(tr_dl):.4f} "
          f"| Val accuracy {acc*100:.2f}% "
          f"| Best {best_acc*100:.2f}% "
          f"| Time {time.time()-start:.1f}s")

resnet.load_state_dict(best_state)
print(f"\nBest validation accuracy: {best_acc*100:.2f}%")


Epoch 1/10 | Loss 0.3243 | Val accuracy 78.00% | Best 78.00% | Time 10.1s
Epoch 2/10 | Loss 0.2925 | Val accuracy 77.75% | Best 78.00% | Time 8.3s
Epoch 3/10 | Loss 0.2757 | Val accuracy 77.00% | Best 78.00% | Time 8.4s
Epoch 4/10 | Loss 0.2551 | Val accuracy 78.25% | Best 78.25% | Time 8.6s
Epoch 5/10 | Loss 0.2512 | Val accuracy 79.25% | Best 79.25% | Time 8.4s
Epoch 6/10 | Loss 0.2412 | Val accuracy 80.50% | Best 80.50% | Time 8.3s
Epoch 7/10 | Loss 0.2374 | Val accuracy 78.25% | Best 80.50% | Time 8.3s
Epoch 8/10 | Loss 0.2346 | Val accuracy 79.25% | Best 80.50% | Time 8.1s
Epoch 9/10 | Loss 0.2324 | Val accuracy 79.75% | Best 80.50% | Time 8.2s
Epoch 10/10 | Loss 0.2324 | Val accuracy 79.50% | Best 80.50% | Time 8.2s

Best validation accuracy: 80.50%


In [51]:

import torch
from sklearn.metrics import accuracy_score, classification_report

resnet.load_state_dict(torch.load(
    "/kaggle/working/best_resnet18_finetuned.pth",
    map_location=device,
    weights_only=True
))
resnet.eval()

preds, targets = [], []
with torch.no_grad():
    for x, y in test_ds:
        x = x.unsqueeze(0).to(device)
        pred = resnet(x).argmax(1).item()
        preds.append(pred)
        targets.append(int(y))

print(f"Fine-tuned ResNet18 Test Accuracy: "
      f"{accuracy_score(targets, preds)*100:.2f}%")
print(classification_report(
    targets, preds,
    labels=[0, 1, 2, 3],
    target_names=[
        "MildDemented", "ModerateDemented",
        "NonDemented", "VeryMildDemented"
    ],
    zero_division=0
))


Fine-tuned ResNet18 Test Accuracy: 81.00%
                  precision    recall  f1-score   support

    MildDemented       0.80      0.86      0.83       100
ModerateDemented       0.97      1.00      0.99       100
     NonDemented       0.79      0.74      0.76       100
VeryMildDemented       0.67      0.64      0.66       100

        accuracy                           0.81       400
       macro avg       0.81      0.81      0.81       400
    weighted avg       0.81      0.81      0.81       400



In [52]:

import torch, copy, time
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

# Stronger augmentation for training only
train_ds.transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Validation/test remain deterministic
val_ds.transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

tr_dl = DataLoader(train_ds, batch_size=32, shuffle=True,
                   num_workers=2, pin_memory=True)
va_dl = DataLoader(val_ds, batch_size=32, shuffle=False,
                   num_workers=2, pin_memory=True)

# Restore the best fine-tuned model
resnet.load_state_dict(torch.load(
    "/kaggle/working/best_resnet18_finetuned.pth",
    map_location=device, weights_only=True
))
resnet = resnet.to(device)

optimizer = torch.optim.AdamW([
    {"params": resnet.layer3.parameters(), "lr": 5e-6},
    {"params": resnet.layer4.parameters(), "lr": 1e-5},
    {"params": resnet.fc.parameters(), "lr": 3e-5}
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.03)

best_acc = 0.0
best_state = copy.deepcopy(resnet.state_dict())

for epoch in range(1, 9):
    start = time.time()
    resnet.train()
    total_loss = 0.0

    for x, y in tr_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(resnet(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    resnet.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in va_dl:
            preds.extend(resnet(x.to(device)).argmax(1).cpu().tolist())
            targets.extend(y.cpu().tolist())

    acc = accuracy_score(targets, preds)
    scheduler.step(acc)

    if acc > best_acc:
        best_acc = acc
        best_state = copy.deepcopy(resnet.state_dict())
        torch.save(best_state,
                   "/kaggle/working/best_resnet18_augmented.pth")

    print(f"Epoch {epoch}/8 | Loss {total_loss/len(tr_dl):.4f} "
          f"| Val {acc*100:.2f}% | Best {best_acc*100:.2f}% "
          f"| {time.time()-start:.1f}s")

resnet.load_state_dict(best_state)
print(f"\nBest validation accuracy: {best_acc*100:.2f}%")


Epoch 1/8 | Loss 0.4547 | Val 75.75% | Best 75.75% | 12.8s
Epoch 2/8 | Loss 0.4142 | Val 76.00% | Best 76.00% | 10.3s
Epoch 3/8 | Loss 0.3786 | Val 76.25% | Best 76.25% | 10.2s
Epoch 4/8 | Loss 0.3877 | Val 75.75% | Best 76.25% | 10.5s
Epoch 5/8 | Loss 0.3519 | Val 76.25% | Best 76.25% | 10.4s
Epoch 6/8 | Loss 0.3431 | Val 76.75% | Best 76.75% | 10.2s
Epoch 7/8 | Loss 0.3400 | Val 78.00% | Best 78.00% | 10.6s
Epoch 8/8 | Loss 0.3237 | Val 77.75% | Best 78.00% | 10.1s

Best validation accuracy: 78.00%


In [53]:

import torch
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

resnet.load_state_dict(torch.load(
    "/kaggle/working/best_resnet18_finetuned.pth",
    map_location=device,
    weights_only=True
))
resnet.eval()

preds, targets = [], []
with torch.no_grad():
    for x, y in va_dl:
        out = resnet(x.to(device))
        preds.extend(out.argmax(1).cpu().tolist())
        targets.extend(y.cpu().tolist())

names = [
    "MildDemented", "ModerateDemented",
    "NonDemented", "VeryMildDemented"
]

print(classification_report(
    targets, preds, labels=[0, 1, 2, 3],
    target_names=names, zero_division=0
))
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(targets, preds, labels=[0, 1, 2, 3]))


                  precision    recall  f1-score   support

    MildDemented       0.73      0.92      0.81       100
ModerateDemented       1.00      1.00      1.00       100
     NonDemented       0.74      0.78      0.76       100
VeryMildDemented       0.76      0.52      0.62       100

        accuracy                           0.81       400
       macro avg       0.81      0.81      0.80       400
    weighted avg       0.81      0.81      0.80       400

Confusion matrix (rows=true, columns=predicted):
[[ 92   0   2   6]
 [  0 100   0   0]
 [ 12   0  78  10]
 [ 22   0  26  52]]


In [54]:

import torch, copy, time
import torch.nn as nn
from sklearn.metrics import accuracy_score

resnet.load_state_dict(torch.load(
    "/kaggle/working/best_resnet18_finetuned.pth",
    map_location=device, weights_only=True
))
resnet = resnet.to(device)

# Give more weight to the classes currently being confused
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, 1.0, 1.3, 1.6], device=device),
    label_smoothing=0.02
)

optimizer = torch.optim.AdamW([
    {"params": resnet.layer3.parameters(), "lr": 5e-6},
    {"params": resnet.layer4.parameters(), "lr": 1e-5},
    {"params": resnet.fc.parameters(), "lr": 3e-5}
], weight_decay=1e-4)

best_acc = 0.0
best_state = copy.deepcopy(resnet.state_dict())

for epoch in range(1, 6):
    resnet.train()
    start = time.time()

    for x, y in tr_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(resnet(x), y)
        loss.backward()
        optimizer.step()

    resnet.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in va_dl:
            preds.extend(resnet(x.to(device)).argmax(1).cpu().tolist())
            targets.extend(y.cpu().tolist())

    acc = accuracy_score(targets, preds)
    if acc > best_acc:
        best_acc = acc
        best_state = copy.deepcopy(resnet.state_dict())
        torch.save(best_state,
                   "/kaggle/working/best_resnet18_weighted.pth")

    print(f"Epoch {epoch}/5 | Val accuracy: {acc*100:.2f}% "
          f"| Best: {best_acc*100:.2f}% "
          f"| Time: {time.time()-start:.1f}s")

resnet.load_state_dict(best_state)
print("Best validation accuracy:", round(best_acc*100, 2), "%")


Epoch 1/5 | Val accuracy: 75.50% | Best: 75.50% | Time: 10.2s
Epoch 2/5 | Val accuracy: 75.25% | Best: 75.50% | Time: 10.1s
Epoch 3/5 | Val accuracy: 75.50% | Best: 75.50% | Time: 9.9s
Epoch 4/5 | Val accuracy: 76.50% | Best: 76.50% | Time: 11.0s
Epoch 5/5 | Val accuracy: 76.50% | Best: 76.50% | Time: 9.8s
Best validation accuracy: 76.5 %


In [55]:

import torch, copy, time
import torch.nn as nn
from sklearn.metrics import accuracy_score

# Restore the best original fine-tuned checkpoint
resnet.load_state_dict(torch.load(
    "/kaggle/working/best_resnet18_finetuned.pth",
    map_location=device, weights_only=True
))
resnet = resnet.to(device)

for p in resnet.parameters():
    p.requires_grad = False
for p in resnet.layer3.parameters():
    p.requires_grad = True
for p in resnet.layer4.parameters():
    p.requires_grad = True
for p in resnet.fc.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam([
    {"params": resnet.layer3.parameters(), "lr": 1e-5},
    {"params": resnet.layer4.parameters(), "lr": 3e-5},
    {"params": resnet.fc.parameters(), "lr": 1e-4}
])

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[1e-5, 3e-5, 1e-4],
    epochs=5,
    steps_per_epoch=len(tr_dl),
    pct_start=0.2
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.02)

best_acc = 0
best_state = copy.deepcopy(resnet.state_dict())

for epoch in range(1, 6):
    start = time.time()
    resnet.train()
    for x, y in tr_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(resnet(x), y)
        loss.backward()
        optimizer.step()
        scheduler.step()

    resnet.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in va_dl:
            preds.extend(resnet(x.to(device)).argmax(1).cpu().tolist())
            targets.extend(y.cpu().tolist())

    acc = accuracy_score(targets, preds)
    if acc > best_acc:
        best_acc = acc
        best_state = copy.deepcopy(resnet.state_dict())
        torch.save(best_state, "/kaggle/working/best_resnet18_onecycle.pth")

    print(f"Epoch {epoch}/5 | Val: {acc*100:.2f}% "
          f"| Best: {best_acc*100:.2f}% "
          f"| Time: {time.time()-start:.1f}s")

resnet.load_state_dict(best_state)
print("Best validation accuracy:", round(best_acc*100, 2), "%")


Epoch 1/5 | Val: 76.50% | Best: 76.50% | Time: 9.8s
Epoch 2/5 | Val: 77.25% | Best: 77.25% | Time: 9.8s
Epoch 3/5 | Val: 76.75% | Best: 77.25% | Time: 9.9s
Epoch 4/5 | Val: 77.75% | Best: 77.75% | Time: 10.1s
Epoch 5/5 | Val: 77.50% | Best: 77.75% | Time: 9.8s
Best validation accuracy: 77.75 %


In [56]:

import os

root = "/kaggle/input/datasets/uraninjo/augmented-alzheimer-mri-dataset"

for folder in ["AugmentedAlzheimerDataset", "OriginalDataset"]:
    path = os.path.join(root, folder)
    print(f"\n{folder}")
    for cls in sorted(os.listdir(path)):
        cls_path = os.path.join(path, cls)
        if os.path.isdir(cls_path):
            print(f"  {cls}: {len(os.listdir(cls_path))} images")



AugmentedAlzheimerDataset
  MildDemented: 8960 images
  ModerateDemented: 6464 images
  NonDemented: 9600 images
  VeryMildDemented: 8960 images

OriginalDataset
  MildDemented: 896 images
  ModerateDemented: 64 images
  NonDemented: 3200 images
  VeryMildDemented: 2240 images


In [58]:

import os
from glob import glob
from sklearn.model_selection import train_test_split
from collections import Counter

root = "/kaggle/input/datasets/uraninjo/augmented-alzheimer-mri-dataset/OriginalDataset"
classes = [
    "MildDemented", "ModerateDemented",
    "NonDemented", "VeryMildDemented"
]

paths, labels = [], []
for idx, cls in enumerate(classes):
    files = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
        files.extend(glob(os.path.join(root, cls, ext)))
    paths.extend(files)
    labels.extend([idx] * len(files))

train_paths_new, temp_paths, train_y_new, temp_y = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=42
)
val_paths_new, test_paths_new, val_y_new, test_y_new = train_test_split(
    temp_paths, temp_y, test_size=0.50, stratify=temp_y, random_state=42
)

print("Train:", len(train_paths_new), Counter(train_y_new))
print("Validation:", len(val_paths_new), Counter(val_y_new))
print("Test:", len(test_paths_new), Counter(test_y_new))


Train: 4480 Counter({2: 2240, 3: 1568, 0: 627, 1: 45})
Validation: 960 Counter({2: 480, 3: 336, 0: 134, 1: 10})
Test: 960 Counter({2: 480, 3: 336, 0: 135, 1: 9})


In [59]:

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class OriginalMRIDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(image), int(self.labels[idx])

train_ds_new = OriginalMRIDataset(train_paths_new, train_y_new, train_tf)
val_ds_new = OriginalMRIDataset(val_paths_new, val_y_new, eval_tf)
test_ds_new = OriginalMRIDataset(test_paths_new, test_y_new, eval_tf)

train_dl_new = DataLoader(train_ds_new, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)
val_dl_new = DataLoader(val_ds_new, batch_size=32, shuffle=False,
                        num_workers=2, pin_memory=True)
test_dl_new = DataLoader(test_ds_new, batch_size=32, shuffle=False,
                         num_workers=2, pin_memory=True)

print("Loaders ready:", len(train_ds_new), len(val_ds_new), len(test_ds_new))


Loaders ready: 4480 960 960


In [60]:

import torch, copy, time
import torch.nn as nn
from torchvision import models
from sklearn.metrics import accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_new = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model_new.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model_new.fc.in_features, 4)
)
model_new = model_new.to(device)

counts = torch.bincount(torch.tensor(train_y_new), minlength=4).float()
weights = counts.sum() / (4 * counts.clamp(min=1))
criterion = nn.CrossEntropyLoss(weight=weights.to(device))

optimizer = torch.optim.AdamW([
    {"params": model_new.layer3.parameters(), "lr": 5e-5},
    {"params": model_new.layer4.parameters(), "lr": 1e-4},
    {"params": model_new.fc.parameters(), "lr": 3e-4}
], weight_decay=1e-4)

scaler = torch.amp.GradScaler(
    "cuda", enabled=(device.type == "cuda")
)

best_acc, best_state, patience = 0.0, None, 0

for epoch in range(1, 9):
    start = time.time()
    model_new.train()
    loss_sum = 0.0

    for x, y in train_dl_new:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type,
                            dtype=torch.float16,
                            enabled=(device.type == "cuda")):
            loss = criterion(model_new(x), y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item()

    model_new.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in val_dl_new:
            with torch.autocast(device_type=device.type,
                                dtype=torch.float16,
                                enabled=(device.type == "cuda")):
                out = model_new(x.to(device))
            preds.extend(out.argmax(1).cpu().tolist())
            targets.extend(y.tolist())

    acc = accuracy_score(targets, preds)
    if acc > best_acc:
        best_acc = acc
        best_state = copy.deepcopy(model_new.state_dict())
        torch.save(best_state, "/kaggle/working/best_original_resnet.pth")
        patience = 0
    else:
        patience += 1

    print(f"Epoch {epoch}/8 | Loss {loss_sum/len(train_dl_new):.3f} "
          f"| Val {acc*100:.2f}% | Best {best_acc*100:.2f}% "
          f"| {time.time()-start:.1f}s")

    if patience >= 3:
        print("Early stopping.")
        break

model_new.load_state_dict(best_state)
print(f"Best validation accuracy: {best_acc*100:.2f}%")


Epoch 1/8 | Loss 1.103 | Val 61.56% | Best 61.56% | 66.2s
Epoch 2/8 | Loss 0.727 | Val 57.50% | Best 61.56% | 14.4s
Epoch 3/8 | Loss 0.611 | Val 68.12% | Best 68.12% | 15.7s
Epoch 4/8 | Loss 0.487 | Val 68.54% | Best 68.54% | 14.5s
Epoch 5/8 | Loss 0.422 | Val 82.50% | Best 82.50% | 14.6s
Epoch 6/8 | Loss 0.356 | Val 81.46% | Best 82.50% | 14.4s
Epoch 7/8 | Loss 0.292 | Val 88.02% | Best 88.02% | 14.5s
Epoch 8/8 | Loss 0.220 | Val 89.06% | Best 89.06% | 14.4s
Best validation accuracy: 89.06%


In [61]:

import torch, copy, time
import torch.nn as nn
from sklearn.metrics import accuracy_score

model_new.load_state_dict(torch.load(
    "/kaggle/working/best_original_resnet.pth",
    map_location=device,
    weights_only=True
))

optimizer = torch.optim.AdamW([
    {"params": model_new.layer3.parameters(), "lr": 1e-5},
    {"params": model_new.layer4.parameters(), "lr": 2e-5},
    {"params": model_new.fc.parameters(), "lr": 5e-5}
], weight_decay=1e-4)

best_acc = 89.06
best_state = copy.deepcopy(model_new.state_dict())

for epoch in range(9, 13):
    start = time.time()
    model_new.train()
    loss_sum = 0.0

    for x, y in train_dl_new:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type,
                            dtype=torch.float16,
                            enabled=(device.type == "cuda")):
            loss = criterion(model_new(x), y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item()

    model_new.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in val_dl_new:
            with torch.autocast(device_type=device.type,
                                dtype=torch.float16,
                                enabled=(device.type == "cuda")):
                out = model_new(x.to(device))
            preds.extend(out.argmax(1).cpu().tolist())
            targets.extend(y.tolist())

    acc = accuracy_score(targets, preds)
    if acc > best_acc:
        best_acc = acc
        best_state = copy.deepcopy(model_new.state_dict())
        torch.save(best_state, "/kaggle/working/best_original_resnet.pth")

    print(f"Epoch {epoch}/12 | Loss {loss_sum/len(train_dl_new):.3f} "
          f"| Val {acc*100:.2f}% | Best {best_acc:.2f}% "
          f"| {time.time()-start:.1f}s")

model_new.load_state_dict(torch.load(
    "/kaggle/working/best_original_resnet.pth",
    map_location=device,
    weights_only=True
))
print(f"Best validation accuracy: {best_acc:.2f}%")


Epoch 9/12 | Loss 0.162 | Val 92.81% | Best 89.06% | 14.5s
Epoch 10/12 | Loss 0.129 | Val 94.48% | Best 89.06% | 15.3s
Epoch 11/12 | Loss 0.115 | Val 93.54% | Best 89.06% | 14.6s
Epoch 12/12 | Loss 0.109 | Val 93.96% | Best 89.06% | 14.5s
Best validation accuracy: 89.06%


In [62]:

import torch
from sklearn.metrics import accuracy_score, classification_report

# The model currently holds epoch 12 weights.
# Restore the best epoch-10 checkpoint by rerunning training if it wasn't saved.
# Since epoch 10 was best, we need its actual weights to evaluate it.
print("The previous loop did not save epoch 10 weights.")
print("Do not evaluate the current model as the best checkpoint yet.")


The previous loop did not save epoch 10 weights.
Do not evaluate the current model as the best checkpoint yet.


In [63]:
import copy
import torch
import torch.nn as nn
from torch.optim import AdamW
from sklearn.metrics import accuracy_score

# 1. Reload the last saved checkpoint
model_new.load_state_dict(
    torch.load("/kaggle/working/best_original_resnet.pth", map_location=device)
)

# 2. Measure the checkpoint's actual validation accuracy
model_new.eval()
preds, labels = [], []

with torch.no_grad():
    for x, y in val_dl_new:
        x = x.to(device)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            out = model_new(x)
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(y.numpy())

best_acc = accuracy_score(labels, preds)
best_state = copy.deepcopy(model_new.state_dict())
best_epoch = "previous checkpoint"

print(f"Starting validation accuracy: {best_acc * 100:.2f}%")

# 3. Continue training with a lower learning rate
optimizer = AdamW([
    {"params": model_new.layer3.parameters(), "lr": 1e-5},
    {"params": model_new.layer4.parameters(), "lr": 2e-5},
    {"params": model_new.fc.parameters(), "lr": 5e-5}
], weight_decay=1e-4)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

for epoch in range(1, 5):
    model_new.train()
    total_loss = 0

    for x, y in train_dl_new:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            out = model_new(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    model_new.eval()
    preds, labels = [], []

    with torch.no_grad():
        for x, y in val_dl_new:
            x = x.to(device)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                out = model_new(x)
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(y.numpy())

    acc = accuracy_score(labels, preds)

    if acc > best_acc:
        best_acc = acc
        best_epoch = epoch
        best_state = copy.deepcopy(model_new.state_dict())
        torch.save(best_state, "/kaggle/working/best_original_resnet.pth")

    print(
        f"Epoch {epoch}: loss={total_loss / len(train_dl_new):.3f}, "
        f"val_acc={acc * 100:.2f}%, best={best_acc * 100:.2f}%"
    )

# 4. Restore the best saved checkpoint
model_new.load_state_dict(best_state)
print(f"\nBest validation accuracy: {best_acc * 100:.2f}%")
print(f"Best checkpoint: {best_epoch}")

Starting validation accuracy: 89.06%
Epoch 1: loss=0.167, val_acc=91.56%, best=91.56%
Epoch 2: loss=0.132, val_acc=93.85%, best=93.85%
Epoch 3: loss=0.121, val_acc=92.40%, best=93.85%
Epoch 4: loss=0.105, val_acc=94.48%, best=94.48%

Best validation accuracy: 94.48%
Best checkpoint: 4


In [64]:
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the best saved model
model_new.load_state_dict(
    torch.load("/kaggle/working/best_original_resnet.pth", map_location=device)
)
model_new.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for x, y in test_dl_new:
        x = x.to(device)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            outputs = model_new(x)

        y_pred.extend(outputs.argmax(1).cpu().numpy())
        y_true.extend(y.numpy())

test_acc = accuracy_score(y_true, y_pred)

print(f"Final Test Accuracy: {test_acc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3],
    target_names=[
        "MildDemented",
        "ModerateDemented",
        "NonDemented",
        "VeryMildDemented"
    ],
    zero_division=0
))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3]))

Final Test Accuracy: 94.69%

Classification Report:
                  precision    recall  f1-score   support

    MildDemented       0.94      0.96      0.95       135
ModerateDemented       1.00      1.00      1.00         9
     NonDemented       0.95      0.96      0.96       480
VeryMildDemented       0.94      0.92      0.93       336

        accuracy                           0.95       960
       macro avg       0.96      0.96      0.96       960
    weighted avg       0.95      0.95      0.95       960

Confusion Matrix:
[[130   0   2   3]
 [  0   9   0   0]
 [  3   0 460  17]
 [  5   0  21 310]]


In [65]:
import os
import torch

SAVE_DIR = "/kaggle/working/classical_checkpoint"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save best model weights
torch.save(
    model_new.state_dict(),
    f"{SAVE_DIR}/best_resnet18.pth"
)

# Save class names and model configuration
import json

metadata = {
    "model": "ResNet18",
    "num_classes": 4,
    "class_names": [
        "MildDemented",
        "ModerateDemented",
        "NonDemented",
        "VeryMildDemented"
    ],
    "best_validation_accuracy": 94.48,
    "test_accuracy": 94.69,
    "input_size": 224
}

with open(f"{SAVE_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Checkpoint saved:", os.listdir(SAVE_DIR))

Checkpoint saved: ['best_resnet18.pth', 'metadata.json']


In [66]:
import shutil

shutil.make_archive(
    "/kaggle/working/classical_checkpoint",
    "zip",
    "/kaggle/working/classical_checkpoint"
)

print("ZIP ready: /kaggle/working/classical_checkpoint.zip")

ZIP ready: /kaggle/working/classical_checkpoint.zip


In [67]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_new = models.resnet18(weights=None)
model_new.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model_new.fc.in_features, 4)
)

checkpoint_path = "/kaggle/working/classical_checkpoint/best_resnet18.pth"

model_new.load_state_dict(
    torch.load(checkpoint_path, map_location=device, weights_only=True)
)
model_new = model_new.to(device)
model_new.eval()

print("Classical ResNet18 checkpoint loaded successfully!")

Classical ResNet18 checkpoint loaded successfully!


In [68]:
import os
import json
import torch

ARTIFACT_DIR = "/kaggle/working/classical_checkpoint"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Save the best checkpoint selected during validation
torch.save(
    model_new.state_dict(),
    f"{ARTIFACT_DIR}/best_resnet18.pth"
)

metadata = {
    "model": "ResNet18",
    "num_classes": 4,
    "class_names": [
        "MildDemented",
        "ModerateDemented",
        "NonDemented",
        "VeryMildDemented"
    ],
    "best_validation_accuracy": 94.48,
    "test_accuracy": 94.69,
    "input_size": 224
}

with open(f"{ARTIFACT_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Saved artifacts:")
print(os.listdir(ARTIFACT_DIR))

Saved artifacts:
['best_resnet18.pth', 'metadata.json']


In [69]:
import os
import shutil

src = "/kaggle/working/classical_checkpoint"
zip_path = "/kaggle/working/classical_artifacts.zip"

shutil.make_archive(zip_path[:-4], "zip", src)

print("Artifact ZIP:", zip_path)
print("Size (MB):", round(os.path.getsize(zip_path) / (1024 * 1024), 2))

Artifact ZIP: /kaggle/working/classical_artifacts.zip
Size (MB): 39.63


In [71]:
import os, json, shutil

OUT = "/kaggle/working/classical_project_artifacts"
os.makedirs(OUT, exist_ok=True)

# Save the best ResNet18 checkpoint
torch.save(
    model_new.state_dict(),
    f"{OUT}/best_resnet18.pth"
)

# Save experiment metadata
metadata = {
    "model": "ResNet18",
    "classes": [
        "MildDemented",
        "ModerateDemented",
        "NonDemented",
        "VeryMildDemented"
    ],
    "input_size": 224,
    "best_validation_accuracy_percent": 94.48,
    "test_accuracy_percent": 94.69,
    "checkpoint_type": "model_state_dict"
}

with open(f"{OUT}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Save the train/validation/test split for reproducibility
split_vars = [
    "train_paths_new", "train_y_new",
    "val_paths_new", "val_y_new",
    "test_paths_new", "test_y_new"
]

splits = {}
for name in split_vars:
    if name in globals():
        value = globals()[name]
        if hasattr(value, "tolist"):
            value = value.tolist()
        splits[name] = value

with open(f"{OUT}/data_splits.json", "w") as f:
    json.dump(splits, f)

# Package the artifacts only (not the entire working directory)
ZIP = shutil.make_archive(OUT, "zip", OUT)

print("Artifacts saved successfully:")
print(os.listdir(OUT))
print("ZIP:", ZIP)
print("ZIP size (MB):", round(os.path.getsize(ZIP) / 1e6, 2))

Artifacts saved successfully:
['data_splits.json', 'best_resnet18.pth', 'metadata.json']
ZIP: /kaggle/working/classical_project_artifacts.zip
ZIP size (MB): 41.59
